# 4 · Query Expansion (Claude)

Expands the 15 benchmark questions into formal legal terminology using **Claude via LangChain**, with a JSON checkpoint so an interrupted run resumes.

Output: `data/processed_data/benchmarking_data_expanded.csv`.

In [ ]:
import json, time
import pandas as pd
from config import config
from indian_marriage_legal_recommender.query_expansion import expand_query

df = pd.read_csv(config.BENCHMARK_CSV)
ckpt = config.PROCESSED_DIR / 'expansion_checkpoint.json'
cache = json.loads(ckpt.read_text()) if ckpt.exists() else {}
print(f'{len(df)} questions · {len(cache)} already expanded')

In [ ]:
for i, row in df.iterrows():
    key = str(i)
    if key in cache:
        continue
    cache[key] = expand_query(row['Question'])
    ckpt.write_text(json.dumps(cache))   # persist after every call
    time.sleep(0.5)                       # gentle on rate limits

df['Expanded_Query'] = [cache[str(i)] for i in range(len(df))]
df.to_csv(config.BENCHMARK_EXPANDED_CSV, index=False)
ckpt.unlink(missing_ok=True)
print('Saved', config.BENCHMARK_EXPANDED_CSV)